In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

df = pd.read_parquet("df_final.parquet")

cols_to_drop = ["_RFBMI5", "WTKG3", "HTM4"]
df = df.drop(columns=cols_to_drop)
df["DIABETE_BIN"].isna().sum()
df = df.dropna(subset=["DIABETE_BIN"])
df["DIABETE_BIN"].value_counts(normalize=True) * 100

,proportion
DIABETE_BIN,
0.0,83.44219
1.0,16.55781


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_recall_curve

from imblearn.over_sampling import SMOTE



target_col = "DIABETE_BIN"
batch_size = 128
epochs = 30
lr = 3e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


X = df.drop(columns=[target_col, "DIABETE4"])
y = df[target_col]

X["year"] = X["year"].astype(int)
X["year"] = X["year"] - X["year"].min()

X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")




print("Before SMOTE:", np.bincount(y_train))

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train))



class HealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, "values") else y,
                              dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = HealthDataset(X_train, y_train)
test_ds = HealthDataset(X_test, y_test)


train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


class DiabetesDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


model = DiabetesDNN(X_train.shape[1]).to(device)



criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=1e-4
)



scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=lr,
    steps_per_epoch=len(train_loader),
    epochs=epochs
)



scaler_amp = torch.cuda.amp.GradScaler()

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(X_batch).view(-1)
            loss = criterion(logits, y_batch)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")



model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)



prec, rec, thresh = precision_recall_curve(all_labels, all_probs)

f1 = (2 * prec * rec) / (prec + rec + 1e-8)

best_idx = np.nanargmax(f1[:-1])
best_thresh = thresh[best_idx]

print("\nBest threshold:", best_thresh)


all_preds = (all_probs > best_thresh).astype(int)


acc = accuracy_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
f1_score_final = f1_score(all_labels, all_preds)

print("\n===== TEST METRICS =====")
print(f"Accuracy : {acc:.4f}")
print(f"ROC AUC  : {auc:.4f}")
print(f"F1 Score : {f1_score_final:.4f}")



torch.save(model.state_dict(), "diabetes_model.pt")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

print("\nSaved model + scaler + features")

Device: cpu
Before SMOTE: [882842 175186]
After SMOTE: [882842 882842]


/tmp/ipykernel_1960/500645257.py:146: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_1960/500645257.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Epoch 1/30 | Loss: 0.5935
Epoch 2/30 | Loss: 0.5749
Epoch 3/30 | Loss: 0.5713
Epoch 4/30 | Loss: 0.5693
Epoch 5/30 | Loss: 0.5683
Epoch 6/30 | Loss: 0.5676
Epoch 7/30 | Loss: 0.5671
Epoch 8/30 | Loss: 0.5668
Epoch 9/30 | Loss: 0.5666
Epoch 10/30 | Loss: 0.5664
Epoch 11/30 | Loss: 0.5664
Epoch 12/30 | Loss: 0.5662
Epoch 13/30 | Loss: 0.5661
Epoch 14/30 | Loss: 0.5661
Epoch 15/30 | Loss: 0.5660
Epoch 16/30 | Loss: 0.5659
Epoch 17/30 | Loss: 0.5658
Epoch 18/30 | Loss: 0.5657
Epoch 19/30 | Loss: 0.5655
Epoch 20/30 | Loss: 0.5655
Epoch 21/30 | Loss: 0.5654
Epoch 22/30 | Loss: 0.5653
Epoch 23/30 | Loss: 0.5651
Epoch 24/30 | Loss: 0.5651
Epoch 25/30 | Loss: 0.5650
Epoch 26/30 | Loss: 0.5649
Epoch 27/30 | Loss: 0.5648
Epoch 28/30 | Loss: 0.5647
Epoch 29/30 | Loss: 0.5647
Epoch 30/30 | Loss: 0.5648

Best threshold: 0.59192944

===== TEST METRICS =====
Accuracy : 0.7463
ROC AUC  : 0.7741
F1 Score : 0.4431

Saved model + scaler + features


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import joblib
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_recall_curve
from imblearn.over_sampling import SMOTE



target_col = "DIABETE_BIN"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)



X = df.drop(columns=[target_col, "DIABETE4"])
y = df[target_col]

X["year"] = X["year"].astype(int)
X["year"] = X["year"] - X["year"].min()

X = X.fillna(X.median(numeric_only=True))

feature_columns = X.columns.tolist()



X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")


smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)



class HealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, "values") else y,
                              dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = HealthDataset(X_train, y_train)
test_ds = HealthDataset(X_test, y_test)


class DiabetesDNN(nn.Module):
    def __init__(self, input_dim, h1, h2, dropout):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, 1)
        )

    def forward(self, x):
        return self.net(x)



def objective(trial):

    h1 = trial.suggest_int("h1", 64, 256)
    h2 = trial.suggest_int("h2", 32, 128)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DiabetesDNN(X_train.shape[1], h1, h2, dropout).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for _ in range(5):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch).view(-1)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    test_loader = DataLoader(test_ds, batch_size=512)

    probs = []
    labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch).view(-1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(y_batch.numpy())

    probs = np.array(probs)
    labels = np.array(labels)

    preds = (probs > 0.5).astype(int)

    return f1_score(labels, preds)



study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("\nBest params:", study.best_params)



best = study.best_params

train_loader = DataLoader(
    train_ds,
    batch_size=best["batch_size"],
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=512,
    shuffle=False
)

model = DiabetesDNN(
    X_train.shape[1],
    best["h1"],
    best["h2"],
    best["dropout"]
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=best["lr"],
    weight_decay=best["weight_decay"]
)

criterion = nn.BCEWithLogitsLoss()



epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch).view(-1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")



model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)


prec, rec, thresh = precision_recall_curve(all_labels, all_probs)
f1 = (2 * prec * rec) / (prec + rec + 1e-8)

best_idx = np.nanargmax(f1[:-1])
best_thresh = thresh[best_idx]

print("\nBest threshold:", best_thresh)

final_preds = (all_probs > best_thresh).astype(int)

print("Final F1:", f1_score(all_labels, final_preds))



torch.save(model.state_dict(), "diabetes_model.pt")

joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
joblib.dump(best, "best_hyperparams.pkl")
joblib.dump(best_thresh, "best_threshold.pkl")

print("\nSaved:")
print("- model (diabetes_model.pt)")
print("- scaler.pkl")
print("- feature_columns.pkl")
print("- best_hyperparams.pkl")
print("- best_threshold.pkl")

Device: cpu


[I 2026-04-21 23:59:38,032] A new study created in memory with name: no-name-35f31ce7-edeb-43cb-a28b-ca3a95e44f97
[I 2026-04-22 00:04:08,644] Trial 0 finished with value: 0.428280142988037 and parameters: {'h1': 184, 'h2': 83, 'dropout': 0.40295710583713007, 'lr': 4.25456676396395e-05, 'weight_decay': 1.532559720591831e-05, 'batch_size': 128}. Best is trial 0 with value: 0.428280142988037.
[I 2026-04-22 00:07:55,979] Trial 1 finished with value: 0.4300884955752212 and parameters: {'h1': 135, 'h2': 120, 'dropout': 0.3556588410484819, 'lr': 0.00039122652768045987, 'weight_decay': 3.754529607476629e-06, 'batch_size': 256}. Best is trial 1 with value: 0.4300884955752212.
[I 2026-04-22 00:13:29,418] Trial 2 finished with value: 0.42746773430205043 and parameters: {'h1': 113, 'h2': 91, 'dropout': 0.2972799596210062, 'lr': 0.000242434288837148, 'weight_decay': 7.141680521050189e-06, 'batch_size': 64}. Best is trial 1 with value: 0.4300884955752212.
[I 2026-04-22 00:18:53,705] Trial 3 finished


Best params: {'h1': 127, 'h2': 77, 'dropout': 0.3587800756016913, 'lr': 0.00039376639505710626, 'weight_decay': 0.0003236049846990099, 'batch_size': 256}
Epoch 1/20 | Loss: 0.5736
Epoch 2/20 | Loss: 0.5691
Epoch 3/20 | Loss: 0.5685
Epoch 4/20 | Loss: 0.5684
Epoch 5/20 | Loss: 0.5683
Epoch 6/20 | Loss: 0.5683
Epoch 7/20 | Loss: 0.5685
Epoch 8/20 | Loss: 0.5682
Epoch 9/20 | Loss: 0.5684
Epoch 10/20 | Loss: 0.5681
Epoch 11/20 | Loss: 0.5683
Epoch 12/20 | Loss: 0.5682
Epoch 13/20 | Loss: 0.5681
Epoch 14/20 | Loss: 0.5681
Epoch 15/20 | Loss: 0.5682
Epoch 16/20 | Loss: 0.5681
Epoch 17/20 | Loss: 0.5682
Epoch 18/20 | Loss: 0.5682
Epoch 19/20 | Loss: 0.5682
Epoch 20/20 | Loss: 0.5681

Best threshold: 0.58248633
Final F1: 0.442715643421422

Saved:
- model (diabetes_model.pt)
- scaler.pkl
- feature_columns.pkl
- best_hyperparams.pkl
- best_threshold.pkl
